# 02. Prétraitement des données (Data Preprocessing)

Ce notebook transforme les données brutes de `data/raw/` en jeux d'entraînement et de test prêts pour les modèles. Il suit la démarche du rapport de TP (chapitre 1 *Raw Data Cleaning* et chapitre 3 *Data Preprocessing*).

| Étape | Contenu |
|-------|---------|
| **0** | Chargement et contrôles qualité |
| **1** | Nettoyage : texte, valeurs bruitées, conversions, binaires, ordinales, variable dérivée, doublons, colonnes inutiles |
| **2** | Cible, colonnes à exclure (fuite d'information) |
| **3** | Valeurs manquantes, encodage one-hot |
| **4** | Découpage train / test |
| **5** | Imputation numérique et normalisation (ajustées sur le train uniquement) |
| **6** | Sauvegarde dans `data/processed/` |

L'exploration (graphiques, corrélations) est faite dans `01_data_understanding_eda.ipynb` et les modèles dans les notebooks suivants.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

# ---------------- Configuration ----------------
DATA_FOLDER   = '../data/raw'
PROCESSED_DIR = '../data/processed'
TARGET        = 'compatible'                      # la colonne à prédire
LEAKAGE_COLS  = ['compatibility_score', 'relationship_longevity_months', 'pair_id']
AGE_MIN, AGE_MAX = 0, 120                         # bornes d'âge plausibles (rapport de TP)
TEST_SIZE     = 0.2
RANDOM_STATE  = 42


def text_columns(frame):
    """Colonnes texte / catégorielles (ni numériques, ni booléennes)."""
    return [c for c in frame.columns
            if not pd.api.types.is_numeric_dtype(frame[c]) and not pd.api.types.is_bool_dtype(frame[c])]

## Étape 0 : Chargement et contrôles qualité

In [ ]:
csv_files = [f for f in os.listdir(DATA_FOLDER) if f.lower().endswith('.csv')]
print('CSV files found:', csv_files)

# On utilise le premier CSV (remplacer par le nom exact s'il y en a plusieurs)
df = pd.read_csv(os.path.join(DATA_FOLDER, csv_files[0]))
raw_shape = df.shape

print('Shape:', df.shape)
print('\nData types and non-null counts:')
df.info()
print('\nDuplicate rows:', df.duplicated().sum())
print('\nMissing values per column:')
print(df.isnull().sum().sort_values(ascending=False))
df.head()

In [ ]:
# --- Contrôles qualité ---
missing = df.isnull().sum()
print('Colonnes avec valeurs manquantes :\n', missing[missing > 0])
print('Total manquantes :', missing.sum())

print('\nDuplicate pair_id:', df['pair_id'].duplicated().sum() if 'pair_id' in df.columns else 'colonne absente')

print('\nPlages de valeurs des colonnes numériques (valeurs impossibles ?) :')
print(df.describe().T[['min', 'max', 'mean']])

for c in text_columns(df):
    print(f'\n{c}: {df[c].nunique()} unique ->', df[c].unique()[:10])

## Étape 1 : Nettoyage

Chaque anomalie est traitée dans un ordre logique. Le code détecte automatiquement les types de colonnes : si une anomalie n'existe pas dans le fichier Cupid, la cellule l'indique simplement.

### 1.1 Standardiser le texte

Une orthographe ou une casse différente (`"Male"`, `"male "`) crée de fausses catégories. On supprime les espaces superflus, on met en minuscules, puis on applique un dictionnaire de corrections optionnel.

In [ ]:
text_cols = text_columns(df)
print('Colonnes texte :', text_cols)

CORRECTIONS = {
    # 'e.coi': 'e.coli',   # exemple : dictionnaire de corrections de typos
}

for col in text_cols:
    df[col] = (df[col].str.strip()
                      .str.replace(r'\s+', ' ', regex=True)
                      .str.lower())
if CORRECTIONS:
    df[text_cols] = df[text_cols].replace(CORRECTIONS)

for col in text_cols:
    n = df[col].nunique()
    if n <= 15:
        print(f'{col} ({n}) :', sorted(df[col].dropna().unique().tolist()))
    else:
        print(f'{col} : {n} valeurs uniques')

### 1.2 Normaliser les valeurs bruitées

Les symboles et mots de remplissage (`?`, `n/a`, `unknown`...) deviennent de vraies valeurs manquantes (`NaN`) pour ne pas être comptés comme des catégories.

In [ ]:
PLACEHOLDERS = ['', '?', '??', 'n/a', 'nan', 'null', 'unknown', 'undefined', '-', '--', 'missing']

before = df.isna().sum().sum()
for col in text_columns(df):
    df[col] = df[col].replace(PLACEHOLDERS, np.nan)
after = df.isna().sum().sum()
print(f'Valeurs bruitées converties en NaN : {after - before}')

### 1.3 Colonnes combinées, nombres en texte, bornes d'âge

(a) Une colonne de type `x/y` est séparée en deux. (b) Les colonnes presque entièrement numériques stockées en texte sont converties. (c) Les âges hors de 0 à 120 sont mis à `NaN`.

In [ ]:
combined_cols = []
for col in text_columns(df):
    s = df[col].dropna().astype(str)
    if len(s) and s.str.fullmatch(r'[^/]+/[^/]+').mean() >= 0.5:
        parts = df[col].str.split('/', expand=True)
        first = pd.to_numeric(parts[0], errors='coerce')
        df[f'{col}_1'] = first if first.notna().mean() > 0.5 else parts[0]
        df[f'{col}_2'] = parts[1]
        combined_cols.append(col)
print('Colonnes combinées séparées :', combined_cols if combined_cols else 'aucune')

converted = []
for col in text_columns(df):
    num = pd.to_numeric(df[col], errors='coerce')
    non_null = df[col].notna().sum()
    if non_null and num.notna().sum() / non_null >= 0.95:
        df[col] = num
        converted.append(col)
print('Colonnes texte converties en numérique :', converted if converted else 'aucune')

age_cols = [c for c in df.columns
            if re.search(r'(^|_)age($|_)', c.lower()) and pd.api.types.is_numeric_dtype(df[c])]
for col in age_cols:
    bad = (df[col] < AGE_MIN) | (df[col] > AGE_MAX)
    df.loc[bad, col] = np.nan
    print(f'{col} : {bad.sum()} valeurs hors bornes mises à NaN | plage {df[col].min()} - {df[col].max()}')

### 1.4 Recoder les variables binaires (Yes / No)

Les représentations synonymes (`1/0`, `True/False`, `Y/N`) sont unifiées en `Yes` / `No`. Les colonnes numériques 0/1 ne sont pas touchées.

In [ ]:
YES = {'yes', 'y', 'true', 't', '1', '1.0'}
NO  = {'no', 'n', 'false', 'f', '0', '0.0'}

def to_yes_no(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    return 'Yes' if x in YES else ('No' if x in NO else np.nan)

binary_recoded = []
for col in df.columns:
    if col == TARGET:
        continue
    if pd.api.types.is_bool_dtype(df[col]):
        df[col] = df[col].map({True: 'Yes', False: 'No'})
        binary_recoded.append(col)
    elif col in text_columns(df):
        vals = {str(v).strip().lower() for v in df[col].dropna().unique()}
        if vals and vals <= (YES | NO):
            df[col] = df[col].apply(to_yes_no)
            binary_recoded.append(col)
print('Colonnes binaires recodées :', binary_recoded if binary_recoded else 'aucune')

### 1.5 Encoder les variables ordinales

Une variable ordinale (ex. *never < rarely < often*) est représentée par des entiers pour conserver l'ordre. Les échelles connues sont détectées automatiquement ; on peut en ajouter dans `CUSTOM_ORDINAL`.

In [ ]:
KNOWN_SCALES = [
    {'never': 0, 'rarely': 1, 'regularly': 2, 'often': 3},
    {'never': 0, 'rarely': 1, 'sometimes': 2, 'often': 3, 'very often': 4, 'always': 5},
    {'very low': 0, 'low': 1, 'medium': 2, 'high': 3, 'very high': 4},
    {'strongly disagree': 0, 'disagree': 1, 'neutral': 2, 'agree': 3, 'strongly agree': 4},
]
CUSTOM_ORDINAL = {
    # 'une_colonne': {'low': 0, 'medium': 1, 'high': 2},
}

ordinal_encoded = {}
for col, mapping in CUSTOM_ORDINAL.items():
    if col in df.columns:
        df[col] = df[col].map(mapping)
        ordinal_encoded[col] = mapping

for col in text_columns(df):
    vals = set(df[col].dropna().unique())
    for scale in KNOWN_SCALES:
        if len(vals) >= 2 and vals <= set(scale):
            df[col] = df[col].map(scale)
            ordinal_encoded[col] = scale
            break

print('Colonnes ordinales encodées :', list(ordinal_encoded) if ordinal_encoded else 'aucune')

### 1.6 Variable dérivée

Équivalent de l'indicateur *MultiResistance* du TP : l'écart d'âge entre les deux partenaires, `age_gap = |a_age - b_age|`.

In [ ]:
if {'a_age', 'b_age'} <= set(df.columns):
    df['age_gap'] = (df['a_age'] - df['b_age']).abs()
    print('age_gap créée. Écart moyen : {:.2f} ans'.format(df['age_gap'].mean()))
else:
    print("Colonnes 'a_age' / 'b_age' absentes : aucune variable dérivée créée.")

### 1.7 Supprimer les doublons

Les doublons gonflent artificiellement certains cas et biaisent les modèles. Cette étape est faite **avant** la suppression des identifiants, pour ne pas fusionner par erreur deux paires différentes qui auraient les mêmes caractéristiques.

In [ ]:
initial = len(df)
n_dup = df.duplicated().sum()
print('Lignes dupliquées :', n_dup)
df = df.drop_duplicates().reset_index(drop=True)
print(f'{initial - len(df)} lignes supprimées.')

if 'pair_id' in df.columns:
    print('pair_id dupliqués restants :', df['pair_id'].duplicated().sum())

### 1.8 Supprimer les colonnes inutiles

Identifiants, noms, e-mails, adresses, notes et dates n'ont aucune valeur prédictive. Les colonnes constantes et les colonnes combinées déjà séparées sont aussi retirées. La cible n'est jamais supprimée.

In [ ]:
META_WORDS = {'id', 'name', 'email', 'address', 'phone', 'note', 'notes', 'comment', 'comments', 'timestamp', 'date'}

irrelevant = [c for c in df.columns
              if c != TARGET and (set(re.split(r'[_\s/]+', c.lower())) & META_WORDS)]
constant   = [c for c in df.columns if c != TARGET and df[c].nunique(dropna=True) <= 1]
irrelevant = list(dict.fromkeys(irrelevant + constant + combined_cols))

print('Colonnes supprimées :', irrelevant if irrelevant else 'aucune')
df = df.drop(columns=[c for c in irrelevant if c in df.columns])
print('Shape brute :', raw_shape, '-> nettoyée :', df.shape)

In [ ]:
# Sauvegarde du jeu nettoyé (nom distinct des fichiers produits par src/)
os.makedirs(PROCESSED_DIR, exist_ok=True)
df.to_csv(os.path.join(PROCESSED_DIR, 'cupid_cleaned.csv'), index=False)
print('Jeu nettoyé sauvegardé : cupid_cleaned.csv')

## Étape 2 : Cible et colonnes à exclure

`compatibility_score`, `relationship_longevity_months` et `pair_id` ne doivent pas servir de variables explicatives : les deux premières sont liées directement au résultat (fuite d'information) et la dernière est un identifiant.

In [ ]:
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))

X = df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns] + [TARGET])   # les indices
y = df[TARGET]                                                                  # la réponse à deviner

for c in LEAKAGE_COLS:
    assert c not in X.columns, f"'{c}' ne doit pas rester dans X"
print('Features shape:', X.shape)

## Étape 3 : Valeurs manquantes catégorielles et encodage one-hot

Les valeurs manquantes des colonnes texte deviennent une catégorie `missing` (aucun calcul appris sur les données, donc pas de fuite). Ensuite les colonnes texte sont transformées en indicatrices 0/1. Les valeurs manquantes numériques sont traitées après le découpage (étape 5).

In [ ]:
cat_cols = text_columns(X)
print('Colonnes catégorielles :', cat_cols)

X[cat_cols] = X[cat_cols].fillna('missing')
for c in cat_cols:
    print(f'{c}: {X[c].nunique()} valeurs ->', X[c].unique()[:8])

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)
print('\nAvant :', X.shape, '-> Après :', X_encoded.shape)
X_encoded.head()

## Étape 4 : Découpage train / test

Découpage stratifié 80 % / 20 % pour conserver la même proportion de compatibles dans les deux jeux.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train :', X_train.shape, '| Test :', X_test.shape)
print('\nProportion dans train :\n', y_train.value_counts(normalize=True).round(3))
print('\nProportion dans test :\n', y_test.value_counts(normalize=True).round(3))

## Étape 5 : Imputation numérique et normalisation

La médiane (imputation) et la moyenne / l'écart-type (normalisation) sont **calculés sur le train uniquement**, puis appliqués au test, afin d'éviter toute fuite d'information.

In [ ]:
# Colonnes numériques d'origine (sans les colonnes 0/1 issues de l'encodage)
num_cols = X.select_dtypes(include='number').columns.tolist()
print('Colonnes à imputer / normaliser :', len(num_cols))
print('Valeurs manquantes numériques (train) :', int(X_train[num_cols].isna().sum().sum()))

imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test_scaled[num_cols]  = imputer.transform(X_test[num_cols])

X_train_scaled[num_cols] = scaler.fit_transform(X_train_scaled[num_cols])
X_test_scaled[num_cols]  = scaler.transform(X_test_scaled[num_cols])

assert not X_train_scaled.isna().any().any() and not X_test_scaled.isna().any().any(), 'Valeurs manquantes restantes'
X_train_scaled[num_cols].describe().T[['mean', 'std']].head()

## Étape 6 : Sauvegarde

Les quatre fichiers ci-dessous sont ceux lus par la suite du projet.

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

X_train_scaled.to_csv(os.path.join(PROCESSED_DIR, 'X_train.csv'), index=False)
X_test_scaled.to_csv(os.path.join(PROCESSED_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(PROCESSED_DIR, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(PROCESSED_DIR, 'y_test.csv'), index=False)

print('Fichiers sauvegardés dans data/processed :', os.listdir(PROCESSED_DIR))
print('X_train :', X_train_scaled.shape, '| X_test :', X_test_scaled.shape)